In [1]:
import numpy as np
from numpy.random import randn
import random

In [29]:
train_data = {
    'хороший': True,
    'круто': True,
    'весело': True,
    'ахуенно': True,
    'ултра хорош': True,
    'супер мега хорош': True,
    'неплохо': True,
    'не грустно': True,
    'очень хорошо': True,
    'счаслив': True,
    'я хорош': True,
    'я охуителен': True,
    'мне хорошо': True,
    'мне не плохо': True,
    'Это очень хорошо': True,
    'хуево': False,
    'насрал': False,
    'плохо': False,
    'не хорошо': False,
    'кринжанул': False,
    'ультра насрал': False,
    'не радует': False,
    'радует': True,
    'грустно': False,
    'весело': True,
}

test_data = {
  'это счастье': True,
  'я хороший': True,
  'я не хороший': False,
  'мне не грустно': True,
  'я очень хорош': True,
  'мне очень грустно': False,
  'это плохо, а не хорошо': False,
}

In [3]:
vocab = list(set([w for text in train_data.keys() for w in text.split(' ')]))
vocab_size = len(vocab)

print('%d unique words found' % vocab_size)

23 unique words found


In [4]:
word_to_idx = { w: i for i, w in enumerate(vocab) }
idx_to_word = { i: w for i, w in enumerate(vocab) }

In [5]:
def createInputs(text):
    inputs = []
    for w in text.split(' '):
        v = np.zeros((vocab_size, 1))
        v[word_to_idx[w]] = 1
        inputs.append(v)
    
    return inputs
def softmax(xs):
    return np.exp(xs) / sum(np.exp(xs))


In [10]:
class RNN:
    def __init__(self, input_size, output_size, hidden_size=64):
        #веса
        self.Whh = randn(hidden_size ,hidden_size)/ 1000
        self.Wxh = randn(hidden_size ,input_size)/ 1000
        self.Why = randn(output_size ,hidden_size)/ 1000
        #микровеса
        self.bh = np.zeros((hidden_size , 1))
        self.by = np.zeros((output_size , 1))
        
    def forward(self, inputs):
        h = np.zeros((self.Whh.shape[0], 1))
        
        self.last_input = inputs
        self.last_hs  = {0: h} 
        
        for i, x in enumerate(inputs):
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            self.last_hs[i + 1] = h 
            
        y = self.Why @ h + self.by
        return y ,h 
    def backpros(self , d_y, learn_rate=2e-1):
        n = len(self.last_input)
        
        d_Why = d_y @ self.last_hs[n].T
        d_by = d_y 
        
        d_Whh = np.zeros(self.Whh.shape)
        d_Wxh = np.zeros(self.Wxh.shape)
        d_bh = np.zeros(self.bh.shape)
        
        d_h =  self.Why.T @ d_y
        
        for t in reversed(range(n)):
            temp = ((1-self.last_hs[t + 1] ** 2) * d_h)
            
            d_bh += temp
            d_Whh += temp @ self.last_hs[t].T
            d_Wxh += temp @ self.last_input[t].T
            
            d_h = self.Whh @ temp
        
        for d in [d_Wxh, d_Whh, d_Why, d_bh, d_by]:
            np.clip(d, -1, 1 ,out =d)
            
        self.Whh -= learn_rate * d_Whh
        self.Wxh -= learn_rate * d_Wxh
        self.Why -= learn_rate * d_Why
        self.bh -= learn_rate * d_bh
        self.by -= learn_rate * d_by

In [11]:
rnn = RNN(vocab_size, 2)

In [14]:
def processData(data ,backprop = True):
    items = list(data.items())
    random.shuffle(items)
    
    loss = 0 
    num_correct = 0 
     
    for x ,y in items:
        inputs = createInputs(x)
        target = int(y)
        
        out, _ = rnn.forward(inputs)
        probs = softmax(out)
        
        loss -= np.log(probs[target])
        num_correct += int(np.argmax(probs) == target)
        
        if backprop:
            d_L_d_y = probs
            d_L_d_y[target] -= 1 
            
            rnn.backpros(d_L_d_y)
    return loss / len(data) , num_correct / len(data)

In [30]:
for epoch in range(1000):
    train_loss , train_acc = processData(train_data)
    
    if epoch % 100 == 99:
        print('--- Epocj %d'%(epoch + 1))
        print('Train:t\Loss %.4f | Acc: %.4f'%(train_loss , train_acc))
            
        test_loss , test_acc = processData(test_data ,backprop=False)
        print('Test:t\Loss %.4f | Acc: %.4f'%(test_loss , test_acc))

KeyError: 'хороший'